# 🔬 Faculté des Sciences Ben M'Sik (FSBM) - Université Hassan II de Casablanca
## Projet de Machine Learning & NLP : Moteur de Recherche Sémantique pour la Production Scientifique

---

### 📋 Présentation du Projet & Livrables Académiques
Ce notebook constitue le **Livrable 3** démontrant le fonctionnement de bout en bout de l'architecture d'ingénierie des données et de recherche sémantique développée pour indexer et valoriser la production scientifique des enseignants-chercheurs de la **Faculté des Sciences Ben M'Sik (FSBM)**.

#### 🎯 Objectifs & Critères Validés :
1. **Robustesse de la collecte (Scraping)** : Extraction des données Google Scholar avec stratégies de mitigation des blocages (reprise automatique par checkpointing, temporisation avec jitter anti-bot, rotation des en-têtes).
2. **Qualité du prétraitement & Structuration propre** : Nettoyage typographique, préservation stricte des caractères accentués français, détection de langue (FR/EN), dédoublonnage et export synchronisé en **JSON** et **Parquet**.
3. **Bonne intégration et utilisation de `zembed-1`** : Modèle d'embedding dense `zeroentropy/zembed-1-embedding` (2560 dimensions) adapté aux représentations asymétriques document/requête.
4. **Pertinence de la démonstration sémantique** : Évaluation multicritère, requêtes thématiques réelles (IA, Santé, IoT, Mathématiques) et comparaison formelle face à une recherche par mots-clés classique.


### ⚙️ 0. Configuration de l'environnement et imports
Vérification des chemins d'accès au projet et importation des bibliothèques scientifiques et NLP.


In [ ]:
import os
import sys
import json
import random
import time
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration de l'affichage
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 11

# Résolution de la racine du projet
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

print(f"[*] Racine du projet : {PROJECT_ROOT}")
sys.path.append(str(PROJECT_ROOT))


---
## 🌐 Étape 1 : Collecte Robuste des Profils Chercheurs FSBM (Google Scholar)

### 🛡️ Stratégie de contournement des blocages Google Scholar
Google Scholar applique des mécanismes de détection de bots très stricts (redirections HTTP 302 vers CAPTCHA, codes 429 Too Many Requests, blocages d'adresses IP).

Pour garantir la pérennité du scraping, notre script `src/scraper.py` implémente :
1. **Idempotence & Checkpointing atomique** : Chaque profil scrapé est immédiatement sauvegardé dans `data/raw/raw_scholar_data.json`. Si une interruption survient, la reprise se fait automatiquement sans réinterroger les profils déjà acquis.
2. **Jitter & Temporisation aléatoire** : Utilisation d'un délai dynamique (`random.uniform(3.0, 7.0)`) entre les requêtes au lieu de pauses fixes facilement identifiables par les heuristiques Google.
3. **Rotation d'en-têtes HTTP (User-Agent)** : Simulation de navigateurs récents (desktop et mobile).
4. **Gestion fine des exceptions** : Détection des erreurs `MaxTriesExceededException`, `DOSException` et alertes CAPTCHA avec refroidissement exponentiel.


In [ ]:
# 1. Inspection de la liste initiale des enseignants-chercheurs de la FSBM
input_faculty_path = PROJECT_ROOT / "data" / "raw" / "input_faculty_list.json"
with open(input_faculty_path, "r", encoding="utf-8") as f:
    faculty_list = json.load(f)

print(f"[+] Total d'enseignants-chercheurs répertoriés à la FSBM : {len(faculty_list)}")
print(f"[*] Exemple d'entrée : {faculty_list[0]}")


In [ ]:
# 2. Chargement et statistiques des données brutes récoltées (raw_scholar_data.json)
raw_data_path = PROJECT_ROOT / "data" / "raw" / "raw_scholar_data.json"
with open(raw_data_path, "r", encoding="utf-8") as f:
    raw_profiles = json.load(f)

total_raw_articles = sum(len(p.get("articles", [])) for p in raw_profiles)
total_citations = sum(p.get("metriques", {}).get("citations_totales", 0) for p in raw_profiles)

print(f"[+] Profils collectés dans le checkpoint : {len(raw_profiles)}")
print(f"[+] Total des publications brutes extraites : {total_raw_articles}")
print(f"[+] Total cumulé des citations FSBM : {total_citations:,}")


In [ ]:
# Aperçu des 5 chercheurs les plus cités de l'échantillon FSBM
top_researchers = sorted(
    raw_profiles,
    key=lambda x: x.get("metriques", {}).get("citations_totales", 0),
    reverse=True
)[:5]

print("Top 5 Chercheurs FSBM (par citations totales) :")
print("-" * 75)
for r in top_researchers:
    m = r.get("metriques", {})
    print(f"• {r.get('nom_complet'):<25} | Citations: {m.get('citations_totales', 0):<6} | h-index: {m.get('h_index', 0):<3} | i10-index: {m.get('i10_index', 0)}")


---
## 🧹 Étape 2 : Prétraitement et Structuration des Données

### 🇲🇦 Défi linguistique & Spécificités de la FSBM
La recherche scientifique à la FSBM (Université Hassan II de Casablanca) est rédigée à la fois en **Français** et en **Anglais**. 

* **Problème identifié dans le code baseline** : Une expression régulière naïve (`[^a-zA-Z0-9...]`) détruisait tous les accents français (`é, è, ê, à, ç, î, ô, ù`), transformant par exemple `"santé et détection"` en `"sant  et d tection"`.
* **Solution apportée** :
  1. Utilisation du bloc Unicode Latin étendu (`À-ſ`) pour préserver l'intégralité des accents.
  2. Décodage HTML (`html.unescape`) pour traiter les résidus du web (`&amp;`, `&eacute;`).
  3. Nettoyage des artefacts de copyright et métadonnées d'éditeurs (Elsevier, IEEE, Springer).
  4. Détection automatique de langue (`fr` vs `en`).
  5. Dédoublonnage d'articles et validation de la longueur minimale des résumés (>= 25 caractères).
  6. Exportation double : format hiérarchique **JSON** et format tabulaire **Parquet** (Livrable 2).


In [ ]:
# Chargement du dataset nettoyé final au format Parquet
parquet_path = PROJECT_ROOT / "data" / "processed" / "processed_abstracts.parquet"
df_papers = pd.read_parquet(parquet_path)

print(f"[+] Dataset Parquet chargé avec succès !")
print(f"[+] Dimensions du dataset : {df_papers.shape[0]} lignes (articles) x {df_papers.shape[1]} colonnes")
df_papers.head(3)


In [ ]:
# Vérification de la conservation des caractères accentués français
french_papers = df_papers[df_papers["lang"] == "fr"]
print(f"[*] Nombre d'articles scientifiques en français identifiés : {len(french_papers)}")
print("-" * 80)
if not french_papers.empty:
    sample_fr = french_papers.iloc[0]
    print(f"Titre : {sample_fr['titre']}")
    print(f"Auteurs : {sample_fr['auteurs_str']}")
    print(f"Extrait du résumé nettoyé :\n{sample_fr['abstract_clean'][:220]}...")


In [ ]:
# Statistiques descriptives et répartition linguistique
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Répartition des langues
lang_counts = df_papers["lang"].value_counts()
axes[0].pie(
    lang_counts,
    labels=[f"Anglais ({lang_counts.get('en', 0)})", f"Français ({lang_counts.get('fr', 0)})"],
    autopct="%1.1f%%",
    colors=["#2563eb", "#10b981"],
    startangle=140,
    explode=(0, 0.1) if len(lang_counts) > 1 else None
)
axes[0].set_title("Répartition Linguistique des Résumés Scientifiques FSBM")

# Distribution du nombre de mots dans les résumés
sns.histplot(df_papers["word_count"], bins=30, kde=True, ax=axes[1], color="#3b82f6")
axes[1].set_title("Distribution du Nombre de Mots par Résumé")
axes[1].set_xlabel("Nombre de mots")
axes[1].set_ylabel("Fréquence")

plt.tight_layout()
plt.show()


---
## 🧠 Étape 3 : Représentation Vectorielle avec `zembed-1` & Indexation ChromaDB

### 📐 Modèle d'Embedding : `zeroentropy/zembed-1-embedding`
Le modèle sélectionné répond aux exigences du projet académique :
- **Architecture** : Modèle de plongement sémantique dense basé sur les Transformers.
- **Dimension de sortie** : $\mathbf{d = 2560}$ dimensions (représentation très riche de l'espace sémantique).
- **Asymétrie document / requête** : Le modèle supporte des invites adaptées :
  - `prompt_name='document'` pour l'encodage des résumés à l'indexation.
  - `prompt_name='query'` pour l'encodage des requêtes utilisateurs lors de la recherche.
- **Métrique de similarité** : Distance cosinus ($1 - \cos(\mathbf{u}, \mathbf{v})$) avec vecteurs normalisés $\ell_2$.

### 🗄️ Base de données vectorielle : ChromaDB
La base ChromaDB locale est persistée dans `data/vector_store/` sous la collection `research_papers_db`.


In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

VECTOR_STORE_DIR = PROJECT_ROOT / "data" / "vector_store"
COLLECTION_NAME = "research_papers_db"
MODEL_NAME = "zeroentropy/zembed-1-embedding"

print(f"[*] Connexion à ChromaDB ({VECTOR_STORE_DIR})...")
chroma_client = chromadb.PersistentClient(path=str(VECTOR_STORE_DIR))
collection = chroma_client.get_collection(name=COLLECTION_NAME)

doc_count = collection.count()
sample_peek = collection.peek(limit=1)
emb_dim = len(sample_peek["embeddings"][0]) if sample_peek["embeddings"] is not None else 0

print(f"[+] Collection ChromaDB chargée : '{COLLECTION_NAME}'")
print(f"[+] Nombre d'articles vectorisés indexés : {doc_count}")
print(f"[+] Dimension des vecteurs (zembed-1) : {emb_dim} dimensions")


In [ ]:
# Chargement du modèle SentenceTransformer pour la recherche sémantique
print(f"[*] Chargement du modèle d'embedding {MODEL_NAME}...")
model = SentenceTransformer(MODEL_NAME, trust_remote_code=True)
print(f"[+] Modèle zembed-1 prêt pour les requêtes !")


---
## 🔍 Étape 4 : Démonstration et Évaluation de la Recherche Sémantique

### 🎯 Conception du Moteur de Recherche Sémantique
Pour une requête utilisateur $Q$ :
1. On encode $Q$ avec `zembed-1` en utilisant l'invite `prompt_name='query'`.
2. On interroge l'index HNSW de ChromaDB pour retrouver les $k$ plus proches voisins selon la distance cosinus.
3. On calcule le score de similarité cosinus : $	ext{Similarité} = 1 - 	ext{Distance}$.
4. On restitue les articles scientifiques pertinents avec leurs métadonnées complètes.


In [ ]:
def search_fsbm_papers(query_text: str, top_k: int = 4):
    """
    Effectue une recherche sémantique multilingue sur la base scientifique FSBM.
    """
    # Encodage de la requête avec l'invite 'query'
    encode_kwargs = {"normalize_embeddings": True}
    if "query" in getattr(model, "prompts", {}):
        encode_kwargs["prompt_name"] = "query"

    query_vector = model.encode(query_text, **encode_kwargs).tolist()

    # Requête dans ChromaDB
    results = collection.query(
        query_embeddings=[query_vector],
        n_results=top_k
    )

    documents = results["documents"][0]
    metadatas = results["metadatas"][0]
    distances = results["distances"][0]

    print(f"\n{'='*85}")
    print(f"🔎 REQUÊTE : "{query_text}"")
    print(f"{'='*85}")

    for idx, (doc, meta, dist) in enumerate(zip(documents, metadatas, distances), start=1):
        similarity_score = 1.0 - dist
        print(f"\n{idx}. [{similarity_score*100:.1f}% Match] {meta.get('titre')}")
        print(f"   👤 Chercheur FSBM : {meta.get('nom_complet')} ({meta.get('chercheur_id')})")
        print(f"   📅 Année : {meta.get('date_publication', 'N/A')} | 📚 Journal : {meta.get('journal', 'N/A')}")
        print(f"   ⭐ Citations : {meta.get('citations', 0)} | 🌐 Langue : {meta.get('lang', 'en').upper()}")
        print(f"   📝 Extrait : {doc[:240]}...")

    return results


In [ ]:
# Test 1 : Intelligence Artificielle & Diagnostic Médical (Recherche en Français)
res1 = search_fsbm_papers("Diagnostic du cancer du sein par apprentissage automatique", top_k=3)


In [ ]:
# Test 2 : Sécurité des Réseaux IoT & Capteurs Sans Fil (Recherche en Anglais)
res2 = search_fsbm_papers("Internet of Things security and intrusion detection in wireless sensor networks", top_k=3)


In [ ]:
# Test 3 : Traitement Automatique du Langage Naturel & Chatbots
res3 = search_fsbm_papers("Chatbot intelligent et modélisation de thématiques par NLP", top_k=3)


In [ ]:
# Test 4 : Mathématiques appliquées & Problèmes non-linéaires
res4 = search_fsbm_papers("Méthodes numériques de régularisation et élasticité non linéaire", top_k=3)


---
## ⚖️ Étape 5 : Preuve de Supériorité Sémantique (Sémantique vs Mots-clés)

Dans les moteurs de recherche traditionnels (lexicaux / BM25 / recherche par sous-chaîne), si l'utilisateur saisit une requête avec des synonymes ou des concepts abstraits qui ne figurent pas exactement dans le texte, le résultat est **vide** ou biaisé.

Dans notre moteur vectoriel basé sur **`zembed-1`**, le sens conceptuel profond est capturé dans l'espace à 2560 dimensions.


In [ ]:
def compare_semantic_vs_lexical(query: str, keyword: str):
    """
    Compare la recherche sémantique vectorielle à une recherche textuelle par mot-clé exact.
    """
    print(f"=== COMPARAISON SUR LA THÉMATIQUE : '{query}' ===")
    
    # 1. Recherche par mot-clé exact (lexicale)
    exact_matches = df_papers[
        df_papers["abstract_clean"].str.contains(keyword, case=False, na=False) |
        df_papers["titre"].str.contains(keyword, case=False, na=False)
    ]
    print(f"\n[A] Recherche lexicale sur le mot exact '{keyword}' :")
    print(f"    -> {len(exact_matches)} articles trouvés.")
    if not exact_matches.empty:
        for _, row in exact_matches.head(2).iterrows():
            print(f"    - {row['titre']} ({row['nom_complet']})")

    # 2. Recherche sémantique vectorielle avec zembed-1
    print(f"\n[B] Recherche sémantique vectorielle sur '{query}' :")
    encode_kwargs = {"normalize_embeddings": True}
    if "query" in getattr(model, "prompts", {}):
        encode_kwargs["prompt_name"] = "query"
    
    q_vec = model.encode(query, **encode_kwargs).tolist()
    sem_res = collection.query(query_embeddings=[q_vec], n_results=3)
    for idx, (meta, dist) in enumerate(zip(sem_res["metadatas"][0], sem_res["distances"][0]), start=1):
        print(f"    {idx}. [Score: {(1-dist)*100:.1f}%] {meta['titre']} ({meta['nom_complet']})")

# Test sur un concept où la terminologie varie
compare_semantic_vs_lexical(
    query="Deep learning architectures for medical image recognition",
    keyword="convolutional"
)


---
## 🗺️ Étape 6 : Visualisation de l'Espace Sémantique des Chercheurs FSBM (PCA 2D)

Pour explorer visuellement l'espace des 2560 dimensions généré par `zembed-1`, nous appliquons une analyse en composantes principales (**PCA**) pour projeter les articles scientifiques dans un espace à 2 dimensions.


In [ ]:
from sklearn.decomposition import PCA

# Récupération de tous les vecteurs et métadonnées indexés dans ChromaDB
all_data = collection.get(include=["embeddings", "metadatas"])
all_embeddings = np.array(all_data["embeddings"])
all_metadatas = all_data["metadatas"]

print(f"[*] Matrice des plongements extraite : {all_embeddings.shape}")

# Réduction dimensionnelle par PCA (2560 -> 2 dimensions)
pca = PCA(n_components=2, random_state=42)
embeddings_2d = pca.fit_transform(all_embeddings)
var_explained = pca.explained_variance_ratio_.sum() * 100

print(f"[+] PCA 2D calculée (Variance expliquée : {var_explained:.2f}%)")

# Création d'un DataFrame de visualisation
top_authors_list = [t[0] for t in df_papers["nom_complet"].value_counts().head(6).items()]

viz_df = pd.DataFrame({
    "x": embeddings_2d[:, 0],
    "y": embeddings_2d[:, 1],
    "chercheur": [m.get("nom_complet", "Autre") if m.get("nom_complet") in top_authors_list else "Autres chercheurs" for m in all_metadatas],
    "titre": [m.get("titre", "") for m in all_metadatas],
    "citations": [m.get("citations", 0) for m in all_metadatas]
})

plt.figure(figsize=(12, 8))
sns.scatterplot(
    data=viz_df,
    x="x",
    y="y",
    hue="chercheur",
    size="citations",
    sizes=(30, 250),
    alpha=0.8,
    palette="tab10"
)
plt.title(f"Cartographie Sémantique de la Production Scientifique FSBM (Projection PCA des vecteurs Zembed-1, d=2560)", fontsize=13)
plt.xlabel("Composante Principale 1")
plt.ylabel("Composante Principale 2")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", title="Chercheurs FSBM")
plt.tight_layout()
plt.show()


---
## 🏁 Conclusion & Bilan d'Évaluation

Le pipeline développé répond avec rigueur à l'ensemble des critères académiques :

| Critère Évalué | Réalisation dans le Projet | Statut |
| :--- | :--- | :---: |
| **1. Robustesse du Scraping** | Script `src/scraper.py` doté de reprise sur checkpoint, temporisation aléatoire (jitter anti-bot 3-7s), gestion des exceptions et arrêt gracieux sans corruption. | ✅ Validé |
| **2. Prétraitement & Structuration** | Script `src/cleaner.py` préservant les accents français (`À-ſ`), éliminant les bruits d'éditeurs, détection de langue FR/EN, exports Parquet et JSON conformes. | ✅ Validé |
| **3. Utilisation de `zembed-1`** | Modèle `zeroentropy/zembed-1-embedding` (2560D) intégré avec succès via `SentenceTransformer`, indexation ChromaDB et gestion asymétrique document/query. | ✅ Validé |
| **4. Pertinence Démonstration** | Démonstration sur requêtes bilingues variées, comparaison formalisée Sémantique vs Mot-clé, et cartographie 2D PCA des thématiques de recherche. | ✅ Validé |
| **5. Couche de Déploiement** | Backend API REST haute performance (**FastAPI** dans `src/api.py`) et interface utilisateur interactive moderne (**Streamlit** dans `app.py`). | ✅ Bonus |

---
*Projet réalisé pour la Faculté des Sciences Ben M'Sik (FSBM) - Université Hassan II de Casablanca.*
